# 🧪 ACTION Chip 测试

验证 agent 在 project 层回复中是否自动插入 `[ACTION:id]` 标记。

覆盖场景：
1. 解释页面功能（应出现所有注册 chip）
2. 明确任务后末尾附 chip
3. 无注册 UI 操作时不插 chip（org 层）

In [ ]:
import sys, os, json, uuid
sys.path.insert(0, os.path.abspath('..'))

from nb_utils import setup_env, get_token, run_agent, print_events
setup_env()

import config
AGENT = 'http://127.0.0.1:8000/copilotkit/admin'
print('Agent :', AGENT)

In [ ]:
USERNAME = 'luke'
PASSWORD = 'jmx931228'

TOKEN, ORG = await get_token(USERNAME, PASSWORD)
PROJECT    = 'onboardceshi'
ROUTE      = f'/org/{ORG}/project/{PROJECT}/model-editor'
print('ORG    :', ORG)
print('PROJECT:', PROJECT)
print('ROUTE  :', ROUTE)

In [ ]:
# model-editor 页注册的 UI 操作（与前端 useRegisterAICapability 对齐）
UI_ACTIONS_MODEL_EDITOR = [
    {"id": "create_model",    "label": "新建模型",  "description": "点击打开新建模型表单"},
    {"id": "select_database", "label": "选择数据库", "description": "点击选择要操作的数据库"},
]

def make_chip_payload(msg, *, ui_actions=None, route=ROUTE, project=PROJECT, org=ORG):
    """构造携带完整页面上下文的 payload（含 currentRoute + UI 操作列表）。"""
    return {
        "threadId":  str(uuid.uuid4()),
        "runId":     str(uuid.uuid4()),
        "state":     {},
        "messages":  [{"id": str(uuid.uuid4()), "role": "user", "content": msg}],
        "tools":     [],
        "forwardedProps": {
            "orgName":      org,
            "projectSlug":  project,
            "currentRoute": route,
        },
        "context": [
            {
                "description": "当前页面信息",
                "value": json.dumps({
                    "route":           route,
                    "pageName":        "模型编辑器",
                    "pageDescription": "管理项目的数据模型结构。左侧栏：数据库选择器 + 模型列表；主区域：选中模型后展示数据记录；右侧抽屉：字段详情与编辑。",
                    "pageWorkflow":    "1. 用数据库选择器选择目标数据库\n2. 在左侧模型列表中点击模型\n3. 点击设置图标打开字段管理抽屉",
                }, ensure_ascii=False),
            },
            {
                "description": "当前 AI 上下文",
                "value": json.dumps({
                    "layer":       "project",
                    "orgName":     org,
                    "projectSlug": project,
                    "availableActions": [
                        "navigate_to_org", "navigate_to_model", "navigate_to_data",
                        "open_create_model", "guide_select_database", "guide_create_model",
                        "list_databases", "list_models",
                    ],
                }, ensure_ascii=False),
            },
            {
                "description": "当前页面可用的 UI 操作（点击 [ACTION:id] chip 可高亮对应元素）",
                "value": json.dumps(ui_actions or [], ensure_ascii=False),
            },
        ],
    }

print('helper ready')

## Case 1：请解释当前页面有哪些功能
期望：回复中包含 `[ACTION:create_model]` 和 `[ACTION:select_database]`

In [ ]:
payload = make_chip_payload('请解释当前页面有哪些功能', ui_actions=UI_ACTIONS_MODEL_EDITOR)
events  = await run_agent(AGENT, TOKEN, payload, org=ORG)
print_events(events)

reply = ''.join(e.get('delta','') for e in events if 'TEXT_MESSAGE_CONTENT' in e.get('type',''))
print('\n── Chip 检查 ──────────────────────────────')
for action_id in ['create_model', 'select_database']:
    found = f'[ACTION:{action_id}]' in reply
    print(f'  [{"✅" if found else "❌"}] [ACTION:{action_id}]')

## Case 2：明确任务 + 末尾附 chip
期望：完成任务描述后，末尾附 `本页可用操作：[ACTION:create_model] [ACTION:select_database]`

In [ ]:
payload = make_chip_payload('如何新建一个模型？', ui_actions=UI_ACTIONS_MODEL_EDITOR)
events  = await run_agent(AGENT, TOKEN, payload, org=ORG)
print_events(events)

reply = ''.join(e.get('delta','') for e in events if 'TEXT_MESSAGE_CONTENT' in e.get('type',''))
print('\n── Chip 检查 ──────────────────────────────')
for action_id in ['create_model', 'select_database']:
    found = f'[ACTION:{action_id}]' in reply
    print(f'  [{"✅" if found else "❌"}] [ACTION:{action_id}]')

## Case 3：无注册 UI 操作时不插 chip
期望：回复中不出现任何 `[ACTION:` 标记

In [ ]:
payload = make_chip_payload('当前页面有什么功能', ui_actions=[])  # 空 UI 操作
events  = await run_agent(AGENT, TOKEN, payload, org=ORG)
print_events(events)

reply = ''.join(e.get('delta','') for e in events if 'TEXT_MESSAGE_CONTENT' in e.get('type',''))
print('\n── Chip 检查 ──────────────────────────────')
no_chip = '[ACTION:' not in reply
print(f'  [{"✅" if no_chip else "❌"}] 无 [ACTION:] 标记（符合预期）')